<div style="margin:1.4em 0; background:#f8fafc; color:#111827; border:1px solid #cbd5e1; border-radius:10px; padding:18px 22px; font-size:15.5px; line-height:1.6; font-family:Segoe UI, Roboto, sans-serif;">
  <div style="font-weight:700; color:#1e293b; margin-bottom:.6rem; font-size:16.5px;">
    The Quantum Blackjack
  </div>
  <p style="margin:0 0 .8rem 0;">
    Great job implementing the classical version of the game! We hope the previous chapters have given you good insight into <b>quantum mechanics</b> and <b>quantum programming</b>.
  </p>
  <p style="margin:0;">
    Now, we are ready to take the next step: implementing <b>Quantum Blackjack</b>.
  </p>
</div>


<div style="margin:1.4em 0; background:#f8fafc; color:#111827; border:1px solid #cbd5e1; border-radius:10px; padding:18px 22px; font-size:15.5px; line-height:1.6; font-family:Segoe UI, Roboto, sans-serif;">
  <div style="font-weight:700; color:#1e293b; margin-bottom:.6rem; font-size:16.5px;">
    ⚛️ Step 1: Implementing the Cards (Quantum Version)
  </div>
  <p style="margin:0 0 .8rem 0;">
    Create a class called <b>QCard</b> to represent a <b>quantum version of a playing card</b>.
    Unlike the classical <code>Card</code>, which holds a single rank and suit, a <code>QCard</code> will exist in a <b>superposition</b> of two possible cards.
  </p>
  <p style="margin:0 0 .8rem 0;">
    Follow these steps carefully:
  </p>
  <ol style="margin:.3rem 0 .8rem 1.4rem; padding:0;">
    <li>Define a class named <code>QCard</code>.</li>
    <li>Inside the constructor, initialize the following attributes:
      <ul style="margin:.2rem 0 .5rem 1.2rem;">
        <li><code>c1</code> and <code>c2</code>: two <b>Card</b> objects representing the possible classical states.</li>
        <li><code>a1</code> and <code>a2</code>: two <b>amplitudes</b> (floats) that determine the quantum probability of each card.
          These must satisfy <code>a1² + a2² = 1</code>.</li>
      </ul>
    </li>
    <li>Implement a method called <code>measure()</code>:
      <ul style="margin:.2rem 0 .5rem 1.2rem;">
        <li>Create a 1-qubit <b>QuantumCircuit</b> using <code>Qiskit</code>.</li>
        <li>Initialize it with amplitudes <code>[a1, a2]</code>.</li>
        <li>Measure the qubit to “collapse” the superposition into one of the two classical cards.</li>
        <li>Return the measured <code>Card</code> object.</li>
      </ul>
    </li>
  </ol>
  <p style="margin:0;">
    After completing this step, your <code>QCard</code> will behave like a real quantum object: 
    it stays in superposition until measured, then collapses into one definite card.
  </p>
</div>


In [5]:
import random
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from math import sqrt

from solutions.model.model import Card, Deck


class QCard:
    def __init__(self, c1: Card, a1: float, c2: Card, a2: float):
        self.c1 = c1
        self.a1 = a1
        self.c2 = c2
        self.a2 = a2

    def measure(self):
        qc = QuantumCircuit(1)
        qc.initialize([self.a1, self.a2], 0)
        qc.measure_all()
        sim = AerSimulator()
        tqc = transpile(qc, sim)
        res = sim.run(tqc, shots=1).result().get_counts()
        if '0' in res:
            self.a1 = 1
            self.a2 = 0
            return self.c1
        self.a1 = 0
        self.a2 = 1
        return self.c2
    

<details style="margin:1.2em 0; border:1px solid #cbd5e1; border-radius:10px; overflow:hidden; font-family:Segoe UI, Roboto, sans-serif;">
  <summary style="cursor:pointer; padding:10px 14px; background:#e2e8f0; color:#1e293b; font-weight:600; font-size:15px;">
    💡 Need help for <code>measure()</code> method? <i>Click to see pseudocode</i>
  </summary>
  <div style="padding:12px 16px; background:#f8fafc; color:#111827; font-size:14.5px; line-height:1.55;">
    <pre style="margin:0; font-size:14px; line-height:1.5; background:#f1f5f9; border:1px solid #e5e7eb; border-radius:6px; padding:10px; overflow:auto; color:#1f2937;">
<code>FUNCTION measure():
    CREATE 1-qubit QuantumCircuit
    INITIALIZE qubit with amplitudes [a1, a2]
    MEASURE the qubit
    RUN circuit on AerSimulator with 1 shot
    IF measurement result is '0':
        SET a1 = 1, a2 = 0
        RETURN c1
    ELSE:
        SET a1 = 0, a2 = 1
        RETURN c2</code></pre>
  </div>
</details>


<div style="margin:1.4em 0; background:#f8fafc; color:#111827; border:1px solid #cbd5e1; border-radius:10px; padding:18px 22px; font-size:15.5px; line-height:1.6; font-family:Segoe UI, Roboto, sans-serif;">
  <div style="font-weight:700; color:#1e293b; margin-bottom:.6rem; font-size:16.5px;">
    ♣️ Step 2: Building the Quantum Deck
  </div>
  <p style="margin:0 0 .8rem 0;">
    Now that we have a <b>QCard</b> to represent quantum superpositions, we need a way to draw them randomly — similar to the classical <code>Deck</code> but with quantum flavor.
    In this step, you’ll implement a class called <b>QDeck</b>.
  </p>
  <p style="margin:0 0 .8rem 0;">
    Follow these steps:
  </p>
  <ol style="margin:.3rem 0 .8rem 1.4rem; padding:0;">
    <li>Define a class named <code>QDeck</code>.</li>
    <li>Implement a method <code>draw()</code> that:
      <ul style="margin:.2rem 0 .5rem 1.2rem;">
        <li>Randomly selects two card ranks and two suits from the standard 52-card set.</li>
        <li>Adjust probabilities for superposition of 2 cards accordingly.</li>
        <li>Creates a <code>QCard</code> object using the selected ranks, suits, and amplitudes.</li>
        <li>Returns the generated <code>QCard</code> instance.</li>
      </ul>
    </li>
  </ol>
  <p style="margin:0;">
    After this step, your <b>QDeck</b> will be able to produce random quantum cards — each existing in a valid superposition of two classical cards.
  </p>
</div>


In [ ]:

class QDeck:
    def draw(self) -> QCard:
        rank1 = random.choice(['A', '2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K'])
        suit1 = random.choice(['Hearts', 'Diamonds', 'Clubs', 'Spades'])
        rank2 = random.choice(['A', '2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K'])
        suit2 = random.choice(['Hearts', 'Diamonds', 'Clubs', 'Spades'])
        a1 = random.random()
        a2 = sqrt(1 - a1**2)
        qcard = QCard(Card(rank1, suit1), a1, Card(rank2, suit2), a2)
        return qcard


52
A Hearts


<details style="margin:1.2em 0; border:1px solid #cbd5e1; border-radius:10px; overflow:hidden; font-family:Segoe UI, Roboto, sans-serif;">
  <summary style="cursor:pointer; padding:10px 14px; background:#e2e8f0; color:#1e293b; font-weight:600; font-size:15px;">
    💡 Not sure about probabilities? <i>Click to see reminder</i>
  </summary>
  <div style="padding:12px 16px; background:#f8fafc; color:#111827; font-size:14.5px; line-height:1.55;">
    <p style="margin:0 0 .5rem 0;">
      In quantum mechanics, each amplitude (<code>a₁</code>, <code>a₂</code>) represents a component of the state vector.
      Their <b>squared magnitudes</b> correspond to probabilities and must always sum to 1:
    </p>
    <pre style="margin:0; font-size:14px; line-height:1.5; background:#f1f5f9; border:1px solid #e5e7eb; border-radius:6px; padding:10px; overflow:auto; color:#1f2937;">
<code>a₁² + a₂² = 1</code></pre>
    <p style="margin:.5rem 0 0 0;">
      This ensures that when you measure the quantum card, one of the two outcomes will always occur with total probability 100%.
      For example, if you pick <code>a₁</code> randomly, you can compute the other as:
    </p>
    <pre style="margin:.3rem 0 0 0; font-size:14px; line-height:1.5; background:#f1f5f9; border:1px solid #e5e7eb; border-radius:6px; padding:10px; overflow:auto; color:#1f2937;">
<code>a₂ = √(1 - a₁²)</code></pre>
  </div>
</details>


<details style="margin:1.2em 0; border:1px solid #cbd5e1; border-radius:10px; overflow:hidden; font-family:Segoe UI, Roboto, sans-serif;">
  <summary style="cursor:pointer; padding:10px 14px; background:#e2e8f0; color:#1e293b; font-weight:600; font-size:15px;">
    💡 Need help with <code>draw()</code> method? <i>Click to see pseudocode</i>
  </summary>
  <div style="padding:12px 16px; background:#f8fafc; color:#111827; font-size:14.5px; line-height:1.55;">
    <pre style="margin:0; font-size:14px; line-height:1.5; background:#f1f5f9; border:1px solid #e5e7eb; border-radius:6px; padding:10px; overflow:auto; color:#1f2937;">
<code>FUNCTION draw():
    SELECT random rank1 and suit1
    SELECT random rank2 and suit2
    GENERATE random amplitude a1 between 0 and 1
    COMPUTE a2 = √(1 - a1²)
    CREATE QCard using:
        - Card(rank1, suit1)
        - amplitude a1
        - Card(rank2, suit2)
        - amplitude a2
    RETURN the created QCard</code></pre>
  </div>
</details>


<div style="margin:1.4em 0; background:#f8fafc; color:#111827; border:1px solid #cbd5e1; border-radius:10px; padding:18px 22px; font-size:15.5px; line-height:1.6; font-family:Segoe UI, Roboto, sans-serif;">
  <div style="font-weight:700; color:#1e293b; margin-bottom:.6rem; font-size:16.5px;">
    🫱 Step 3: Managing Quantum Hands
  </div>
  <p style="margin:0 0 .8rem 0;">
    In this step, you will implement a class called <b>Hand</b> to represent a player’s collection of cards — both classical and quantum.
    This class will extend the classical Blackjack logic to handle quantum behavior like <b>measurement</b> and <b>entanglement</b>.
  </p>

  <p style="margin:0 0 .8rem 0;">
    Follow these instructions carefully:
  </p>

  <ol style="margin:.3rem 0 .8rem 1.4rem; padding:0;">
    <li>Define a class named <code>Hand</code> with an attribute <code>cards</code> — a list that stores both <b>Card</b> and <b>QCard</b> objects.</li>
    <li>Implement helper methods:
      <ul style="margin:.2rem 0 .5rem 1.2rem;">
        <li><code>add_card(card)</code>: adds a card (classical or quantum) to the hand.</li>
        <li><code>best_value()</code>: computes the total hand value using Blackjack rules (Aces can be 1 or 11), but raises an error if unmeasured quantum cards remain.</li>
        <li><code>is_blackjack()</code> and <code>is_bust()</code>: evaluate win and loss conditions after measurement.</li>
        <li><code>clear_hand()</code>: clears the hand if are being played</li>
      </ul>
    </li>
    <li>Implement <code>measure_all()</code>:
      <ul style="margin:.2rem 0 .5rem 1.2rem;">
        <li>Iterate over all cards in the hand.</li>
        <li>If a card is a <b>QCard</b>, call its <code>measure()</code> method to collapse it into a classical <b>Card</b>.</li>
        <li>Replace the hand’s cards with the measured results.</li>
      </ul>
    </li>
    <li>Implement <code>entangle_and_measure(card1, card2)</code>:
      <ul style="margin:.2rem 0 .5rem 1.2rem;">
        <li>Take two indices as input and validate them.</li>
        <li>Build a two-qubit <b>QuantumCircuit</b>, initialize each qubit with the amplitudes of the chosen cards, and apply a <b>controlled-X (CX)</b> gate to entangle them.</li>
        <li>Measure both qubits, update the respective cards based on outcomes, and finally call <code>measure_all()</code> to collapse the hand.</li>
      </ul>
    </li>
  </ol>

  <p style="margin:0;">
    After this step, your <b>Hand</b> class will bridge classical Blackjack logic and quantum mechanics — supporting both individual and entangled card measurements.
  </p>
</div>


In [6]:
class Hand:
    def __init__(self):
        self.cards = []

    def add_card(self, card):
        self.cards.append(card)

    def best_value(self):
        if any(isinstance(card, QCard) for card in self.cards):
            raise ValueError("Cannot compute best value with unmeasured quantum cards.")
        # Calculate the best total value of the hand, considering Aces as 1 or 11
        total = sum(card.value() for card in self.cards)
        aces = sum(1 for card in self.cards if card.rank == 'A')
        while total > 21 and aces:
            total -= 10
            aces -= 1
        return total

    def is_blackjack(self):
        if any(isinstance(card, QCard) for card in self.cards):
            raise ValueError("Cannot compute blackjack with unmeasured quantum cards.")
        return len(self.cards) == 2 and self.best_value() == 21

    def is_bust(self):
        if any(isinstance(card, QCard) for card in self.cards):
            raise ValueError("Cannot compute bust with unmeasured quantum cards.")
        return self.best_value() > 21

    def measure_all(self):
        measured_cards = []
        for qcard in self.cards:
            if isinstance(qcard, QCard):
                measured_cards.append(qcard.measure())
            else:
                measured_cards.append(qcard)
        self.cards = measured_cards
        return measured_cards
    
    def entangle_and_measure(self, card1: int, card2: int):
        if card1 < 0 or card1 >= len(self.cards) or card2 < 0 or card2 >= len(self.cards):
            raise IndexError("Card index out of range.")
        if not isinstance(self.cards[card1], QCard) or not isinstance(self.cards[card2], QCard):
            raise ValueError("Both cards must be quantum cards to entangle.")
        qc = QuantumCircuit(2)
        qc.initialize([self.cards[card1].a1, self.cards[card1].a2], 0)
        qc.initialize([self.cards[card2].a1, self.cards[card2].a2], 1)
        qc.cx(0, 1)  # Entangling operation
        qc.measure_all()
        sim = AerSimulator()
        tqc = transpile(qc, sim)
        res = sim.run(tqc, shots=1).result().get_counts()
        if list(res.keys())[0].startswith('0'):
            self.cards[card1] = self.cards[card1].c1
        else:
            self.cards[card1] = self.cards[card1].c2
        if list(res.keys())[0].endswith('0'):
            self.cards[card2] = self.cards[card2].c1
        else:
            self.cards[card2] = self.cards[card2].c2
        return self.measure_all()

<details style="margin:1.2em 0; border:1px solid #cbd5e1; border-radius:10px; overflow:hidden; font-family:Segoe UI, Roboto, sans-serif;">
  <summary style="cursor:pointer; padding:10px 14px; background:#e2e8f0; color:#1e293b; font-weight:600; font-size:15px;">
    ⚠️ Be careful when implementing <code>is_bust()</code> and <code>is_blackjack()</code>
  </summary>
  <div style="padding:12px 16px; background:#f8fafc; color:#111827; font-size:14.5px; line-height:1.55;">
    <p style="margin:0 0 .6rem 0;">
      Remember: quantum cards are still in <b>superposition</b> until you measure them.
      You cannot reliably compute a total hand value or check for Blackjack while any card remains unmeasured.
    </p>
    <ul style="margin:.2rem 0 .6rem 1.4rem;">
      <li>Before calling <code>best_value()</code>, <code>is_bust()</code>, or <code>is_blackjack()</code>, make sure all cards are measured.</li>
      <li>If your hand still contains <code>QCard</code> objects, raise a clear error like:<br>
        <code>ValueError("Cannot compute hand value with unmeasured quantum cards.")</code></li>
    </ul>
    After the measurement, all the cards will be classical and you can do these operations classicaly.
    <p style="margin:0;">
      ✅ Tip: Use your <code>measure_all()</code> method before evaluating the final result to avoid inconsistent logic.
    </p>
  </div>
</details>


<details style="margin:1.2em 0; border:1px solid #cbd5e1; border-radius:10px; overflow:hidden; font-family:Segoe UI, Roboto, sans-serif;">
  <summary style="cursor:pointer; padding:10px 14px; background:#e2e8f0; color:#1e293b; font-weight:600; font-size:15px;">
    🧩 Not sure how to tell if a card is quantum or classical? <i>Click to see tip</i>
  </summary>
  <div style="padding:12px 16px; background:#f8fafc; color:#111827; font-size:14.5px; line-height:1.55;">
    <p style="margin:0 0 .6rem 0;">
      You can use Python’s built-in <code>isinstance()</code> function to check whether a card is a <b>QCard</b> (quantum)
      or a regular <b>Card</b> (classical). This is especially useful inside loops like <code>measure_all()</code> or <code>best_value()</code>.
    </p>
    <pre style="margin:0; font-size:14px; line-height:1.5; background:#f1f5f9; border:1px solid #e5e7eb; border-radius:6px; padding:10px; overflow:auto; color:#1f2937;">
<code>FOR each card IN self.cards:
    IF isinstance(card, QCard):
        // It's a quantum card — call measure() before using its value
    ELSE:
        // It's a classical card — safe to use directly</code></pre>
    <p style="margin:.6rem 0 0 0;">
      ✅ Using <code>isinstance()</code> helps keep your hand logic safe from mixing measured and unmeasured cards.
    </p>
  </div>
</details>


<details style="margin:1.2em 0; border:1px solid #cbd5e1; border-radius:10px; overflow:hidden; font-family:Segoe UI, Roboto, sans-serif;">
  <summary style="cursor:pointer; padding:10px 14px; background:#e2e8f0; color:#1e293b; font-weight:600; font-size:15px;">
    💫 Need help with the entanglement circuit? <i>Click to see explanation</i>
  </summary>
  <div style="padding:12px 16px; background:#f8fafc; color:#111827; font-size:14.5px; line-height:1.55;">
    <p style="margin:0 0 .6rem 0;">
      In the <code>entangle_and_measure()</code> method, we use a <b>two-qubit quantum circuit</b> to create correlation between two quantum cards.
      This simulates the idea that measuring one card can influence the state of the other.
    </p>
    <pre style="margin:0 0 .6rem 0; font-size:14px; line-height:1.5; background:#f1f5f9; border:1px solid #e5e7eb; border-radius:6px; padding:10px; overflow:auto; color:#1f2937;">
<code>FUNCTION entangle_and_measure(card1, card2):
    CREATE QuantumCircuit(2)
    INITIALIZE qubit 0 with [a1, a2] from first card
    INITIALIZE qubit 1 with [a1, a2] from second card
    APPLY CX gate (control: qubit 0, target: qubit 1)
    MEASURE both qubits
    UPDATE both cards based on outcomes
    CALL measure_all() to finalize collapse</code></pre>
    <p style="margin:0;">
      💡 The <b>CX (Controlled-X)</b> gate is the key operation that entangles the two qubits —
      once applied, their measurement outcomes become statistically linked.
      For example, if one collapses to <code>c1</code>, the other might be more likely to collapse to a correlated card.
    </p>
  </div>
</details>


<div style="margin:1.4em 0; background:#f8fafc; color:#111827; border:1px solid #cbd5e1; border-radius:10px; padding:18px 22px; font-size:15.5px; line-height:1.6; font-family:Segoe UI, Roboto, sans-serif;">
  <div style="font-weight:700; color:#1e293b; margin-bottom:.6rem; font-size:16.5px;">
    🧍 Step 4: Implementing Player and Dealer
  </div>
  <p style="margin:0 0 .8rem 0;">
    Our <b>Player</b> and <b>Dealer</b> classes will function exactly as in the classical version of Blackjack.  
    There is no need for quantum-specific modifications here — they simply draw cards, decide actions, and manage their hands.
  </p>
  <p style="margin:0 0 .8rem 0;">
    You can safely <b>copy and paste</b> your <code>Player</code> and <code>Dealer</code> implementations from the previous notebook. (of course, do not forget to adjust classes for new deck class and cards but that is a minor modification)
  </p>
  <ul style="margin:.3rem 0 .8rem 1.4rem; padding:0;">
    <li><b>Player</b> should:
      <ul style="margin:.2rem 0 .5rem 1.2rem;">
        <li>Contain a <code>Hand</code> object.</li>
        <li>Use <code>draw()</code> to pull new cards from the deck (now possibly quantum).</li>
        <li>Use <code>decide()</code> to ask for input (<code>hit</code> or <code>stand</code>).</li>
      </ul>
    </li>
    <li><b>Dealer</b> should:
      <ul style="margin:.2rem 0 .5rem 1.2rem;">
        <li>Inherit from <code>Player</code>.</li>
        <li>Implement <code>play()</code> to draw until reaching a value of 17 or higher.</li>
      </ul>
    </li>
  </ul>
  <p style="margin:0;">
    ✅ These two classes will integrate seamlessly with your <b>QDeck</b> and <b>Hand</b> logic to support both classical and quantum cards.
  </p>
</div>


In [ ]:
class Player:
    def __init__(self, name):
        self.name = name
        self.hand = Hand()

    def draw(self, deck):
        card = deck.draw()
        if card:
            self.hand.add_card(card)
        return card

    def decide(self):
        # Ask for input (hit/stand)
        action = input(f"{self.name}, do you want to hit or stand? ").strip().lower()
        while action not in ['hit', 'stand']:
            print("Invalid input. Please enter 'hit' or 'stand'.")
            action = input(f"{self.name}, do you want to hit or stand? ").strip().lower()
        return action

class Dealer(Player):
    def play(self, deck):
        # Dealer keeps hitting until 17 or higher
        while self.hand.best_value() < 17:
            self.draw(deck)
            print(f"{self.name} drew a card.")

5 Hearts
Invalid input. Please enter 'hit' or 'stand'.
Invalid input. Please enter 'hit' or 'stand'.


<div style="margin:1.4em 0; background:#f8fafc; color:#111827; border:1px solid #cbd5e1; border-radius:10px; padding:18px 22px; font-size:15.5px; line-height:1.6; font-family:Segoe UI, Roboto, sans-serif;">
  <div style="font-weight:700; color:#1e293b; margin-bottom:.6rem; font-size:16.5px;">
    🎮 Step 5: Implementing the Quantum Game Controller
  </div>
  <p style="margin:0 0 .8rem 0;">
    The <b>QGame</b> class acts as the main controller for your Quantum Blackjack game.
    Structurally, it’s almost identical to the <b>classical</b> game implementation you wrote earlier — the only difference is that the player now draws <b>quantum cards</b> from a <code>QDeck</code>.
  </p>
  <p style="margin:0 0 .8rem 0;">
    You can therefore reuse most of your classical logic from the previous notebook and adapt it slightly:
  </p>

  <ol style="margin:.3rem 0 .8rem 1.4rem; padding:0;">
    <li>In <code>__init__()</code>:
      <ul style="margin:.2rem 0 .5rem 1.2rem;">
        <li>Create both a <code>QDeck</code> (for the quantum player) and a <code>Deck</code> (for the classical dealer).</li>
        <li>Initialize <code>Player</code> and <code>Dealer</code> objects just as before.</li>
      </ul>
    </li>
    <li>In <code>deal_initial()</code>:
      <ul style="margin:.2rem 0 .5rem 1.2rem;">
        <li>Use <code>QDeck.draw()</code> when giving cards to the player.</li>
        <li>Keep using <code>Deck.draw()</code> for the dealer’s cards.</li>
        <li>Print amplitudes to show the player’s quantum superposition.</li>
      </ul>
    </li>
    <li>In <code>player_turn()</code> and <code>dealer_turn()</code>:
      <ul style="margin:.2rem 0 .5rem 1.2rem;">
        <li>The structure (hit/stand decisions and dealer behavior) stays the same as before.</li>
        <li>The only addition: after the player stands, call <code>measure_all()</code> to collapse the player’s quantum hand before scoring.</li>
      </ul>
    </li>
    <li>In <code>determine_winner()</code>:
      <ul style="margin:.2rem 0 .5rem 1.2rem;">
        <li>Comparison logic is identical to the classical version — just make sure both hands are fully measured before evaluation.</li>
      </ul>
    </li>
  </ol>

  <p style="margin:0;">
    ✅ Essentially, <b>QGame</b> preserves your classical game flow while introducing quantum draws and measurements.
    This design helps highlight the differences between probabilistic and quantum uncertainty within the same familiar structure.
  </p>
</div>


In [ ]:
class Game:
    def __init__(self):
        self.deck = Deck()
        self.deck.shuffle()
        self.player = Player("Player")
        self.dealer = Dealer("Dealer")

    def deal_initial(self):
        for _ in range(2):
            card = self.player.draw(self.deck)
            print(f"{self.player.name} drew {card.rank} of {card.suit}. Current value: {self.player.hand.best_value()}")
            self.dealer.draw(self.deck)
        print(f"{self.player.name}'s initial hand value: {self.player.hand.best_value()}")
        print(f"{self.dealer.name}'s visible card: {self.dealer.hand.cards[0].rank} of {self.dealer.hand.cards[0].suit}")
        if self.player.hand.is_blackjack():
            print(f"{self.player.name} has a blackjack! {self.player.name} wins!")

    def player_turn(self):
        action = self.player.decide()
        while action == 'hit':
            card = self.player.draw(self.deck)
            print(f"{self.player.name} drew {card.rank} of {card.suit}. Current value: {self.player.hand.best_value()}")
            if self.player.hand.is_bust():
                print(f"{self.player.name} busts with value {self.player.hand.best_value()}!")
                return
            elif self.player.hand.best_value() == 21:
                return
            action = self.player.decide()
        # Player stands
        print(f"{self.player.name} stands with value {self.player.hand.best_value()}.")

    def dealer_turn(self):
        self.dealer.play(self.deck)
        print(f"{self.dealer.name} stands with value {self.dealer.hand.best_value()}.")

    def determine_winner(self):
        player_value = self.player.hand.best_value()
        dealer_value = self.dealer.hand.best_value()

        if self.player.hand.is_bust():
            print("Dealer wins! Player busted.")
        elif self.dealer.hand.is_bust():
            print("Player wins! Dealer busted.")
        elif player_value > dealer_value:
            print("Player wins!")
        elif dealer_value > player_value:
            print("Dealer wins!")
        else:
            print("It's a tie!")

    def play_round(self):
        self.player_turn()
        if not self.player.hand.is_bust():
            self.dealer_turn()

    def start(self):
        self.deal_initial()
        if not self.player.hand.is_blackjack():
            self.play_round()
            self.determine_winner()

game = Game()
game.start()

<details style="margin:1.2em 0; border:1px solid #cbd5e1; border-radius:10px; overflow:hidden; font-family:Segoe UI, Roboto, sans-serif;">
  <summary style="cursor:pointer; padding:10px 14px; background:#e2e8f0; color:#1e293b; font-weight:600; font-size:15px;">
    🔁 Need a reminder about the classical game structure? <i>Click to review similarities</i>
  </summary>
  <div style="padding:12px 16px; background:#f8fafc; color:#111827; font-size:14.5px; line-height:1.55;">
    <p style="margin:0 0 .6rem 0;">
      Your classical <code>Game</code> class already defines the main structure for Blackjack:
      initialization → dealing → player turn → dealer turn → determining the winner.
      The quantum version (<code>QGame</code>) follows the same flow with just a few key differences:
    </p>
    <ul style="margin:.2rem 0 .6rem 1.4rem;">
      <li><b>Player draws</b> cards from <code>QDeck</code> (quantum superpositions).</li>
      <li><b>Dealer draws</b> from the classical <code>Deck</code> (unchanged).</li>
      <li><b>After the player stands</b>, you must call <code>measure_all()</code> to collapse the superposed quantum cards before computing hand values.</li>
      <li><b>Win, bust, and comparison logic</b> all stay the same — just ensure cards are measured first.</li>
    </ul>
    <p style="margin:0;">
      ✅ Keep the structure and logic from your classical version — only modify the <b>card drawing</b> and <b>measurement</b> parts for the quantum twist.
    </p>
  </div>
</details>


<details style="margin:1.2em 0; border:1px solid #cbd5e1; border-radius:10px; overflow:hidden; font-family:Segoe UI, Roboto, sans-serif;">
  <summary style="cursor:pointer; padding:10px 14px; background:#e2e8f0; color:#1e293b; font-weight:600; font-size:15px;">
    💡 Migration checklist: <code>Game</code> ➜ <code>QGame</code>
  </summary>
  <div style="padding:12px 16px; background:#f8fafc; color:#111827; font-size:14.5px; line-height:1.55;">
    <ul style="margin:.2rem 0 .6rem 1.2rem;">
      <li>In <code>__init__()</code>: create both <code>self.qdeck = QDeck()</code> and a classical <code>Deck()</code> for the dealer.</li>
      <li>In <code>deal_initial()</code>: use <code>QDeck.draw()</code> for the player, <code>Deck.draw()</code> for the dealer, and print quantum amplitudes for each draw.</li>
      <li>In <code>player_turn()</code>: logic (hit/stand loop) is the same — just add <code>measure_all()</code> after the player stands.</li>
      <li>In <code>determine_winner()</code>: comparison rules remain identical; ensure all cards are measured before calculating values.</li>
      <li>Leave <code>Dealer.play()</code> unchanged — the dealer still follows classical Blackjack rules (draw until 17).</li>
    </ul>
    <p style="margin:0;">
      ⚠️ If you encounter the error <code>ValueError("Cannot compute hand value with unmeasured quantum cards.")</code>,  
      make sure you’ve measured the player’s hand before determining the winner.
    </p>
  </div>
</details>


<div style="margin:1.2em 0; background:#f8fafc; color:#111827; border:1px solid #cbd5e1; border-radius:10px; padding:18px 22px; font-size:15.5px; line-height:1.55; font-family:Segoe UI, Roboto, sans-serif;">
  <div style="font-weight:700; color:#1e293b; margin-bottom:.6rem; font-size:17px;">
    📦 Putting It All Together — The Game Loop
  </div>
  <p style="margin:0 0 .8rem 0;">
    You’ve now implemented all the building blocks of Blackjack: <b>Card</b>, <b>Deck</b>, <b>Hand</b>, <b>Player</b>, <b>Dealer</b>, and <b>Game</b>.  
    To again play the game with all the money elements, you can just simply copy them from the classical game. Luckly enough your money will not be in a superposition.
  </p>

</div>


```python
```python
def main():

    game = Game()  # create a new game
    game.initialize_game()

    while True:
        if game.money > 0:
            bet = int(input("How much money do you want to bet?"))
            while not game.money >= bet:
                bet = int(input("You dont have that amount of money, how much do you want to bet?"))
            game.start_round(bet)
            answer = input("You have $" + str(game.money) + " left, do you want to play again? (yes/no)").lower()
            if answer != 'yes':
                print("Thank you for playing! The total amount of money you have is: $" + str(game.money))
                break
            print("\n--- New Round ---")
            game.reset()


# Start the game loop
if __name__ == "__main__":
    main()



In [ ]:
def main():

    game = Game()  # create a new game
    game.initialize_game()

    while True:
        if game.money > 0:
            bet = int(input("How much money do you want to bet?"))
            while not game.money >= bet:
                bet = int(input("You dont have that amount of money, how much do you want to bet?"))
            game.start_round(bet)
            answer = input("You have $" + str(game.money) + " left, do you want to play again? (yes/no)").lower()
            if answer != 'yes':
                print("Thank you for playing! The total amount of money you have is: $" + str(game.money))
                break
            print("\n--- New Round ---")
            game.reset()


# Start the game loop
if __name__ == "__main__":
    main()


<div style="margin:1.2em 0; background:#f8fafc; color:#111827; border:1px solid #cbd5e1; border-radius:10px; padding:16px 20px; font-size:15.5px; line-height:1.55; font-family:Segoe UI, Roboto, sans-serif;">
  <div style="font-weight:700; color:#1e293b; margin-bottom:.5rem; font-size:16.5px;">
    🎲 Bonus — Implementing the Tunneling Function
  </div>
  <p style="margin:0 0 .6rem 0;">
    The <b>tunneling()</b> function introduces a bonus mechanic that simulates a quantum “tunneling” effect — giving the player a small chance to recover from a bust. 
    Each point above 21 is treated as a barrier that may be passed with probability <code>p</code>. Passing a barrier decreases the player’s total by one; failure stops the process.
  </p>
  <ol style="margin:.2rem 0 0 1.4rem; padding:0;">
    <li>Define a function <code>tunneling(hand: int, p: float)</code> that takes the player’s total and per-barrier probability <code>p</code>.</li>
    <li>Compute <code>n_of_walls = hand - 21</code> and initialize <code>current_hand = hand</code>.</li>
    <li>Iterate over each wall:
      <ul style="margin:.2rem 0 0 1.2rem; padding:0;">
        <li>Generate a random number; if ≤ <code>p</code>, print the tunneling step and decrease <code>current_hand</code> by 1.</li>
        <li>If the random number is greater than <code>p</code>, print that tunneling stopped and break the loop.</li>
      </ul>
    </li>
    <li>After the loop:
      <ul style="margin:.2rem 0 0 1.2rem; padding:0;">
        <li>If <code>current_hand == 21</code>, print <b>"You tunneled exactly to 21! Congrats."</b></li>
        <li>Otherwise, print <b>"You busted! Dealer wins."</b></li>
      </ul>
    </li>
    <li>This function models the geometric-stop tunneling distribution, where each successful pass occurs with independent probability <code>p</code> until reflection.</li>
  </ol>
</div>


In [ ]:
import random

def tunneling(hand: int, p : float = 0.2 ):
    print("You busted! You are at", hand, "tunneling is activated.")
    n_of_walls = hand - 21
    current_hand = hand

    for i in range(n_of_walls):
        if random.random() <= p:
            print(f"Tunneled through wall {i+1}: {current_hand} → {current_hand - 1}")
            current_hand -= 1
        else:
            print(f"Stopped at wall {i+1}. Tunneling failed!.")
            break

    if current_hand == 21:
        print("You tunneled exactly to 21! Congrats. You win!")
    else:
        print("You busted! Dealer wins.")


tunneling(22)


Stopped at wall 1. Tunneling failed!.
You busted! Dealer wins.


<div style="margin:1.2em 0; background:#f8fafc; color:#111827; border:1px solid #cbd5e1; border-radius:10px; padding:16px 20px; font-size:15.5px; line-height:1.55; font-family:Segoe UI, Roboto, sans-serif;">
  <div style="font-weight:700; color:#1e293b; margin-bottom:.5rem; font-size:16.5px;">
    ⚙️ Integration — Using the Tunneling Mechanic in the Game Engine
  </div>
  <p style="margin:0 0 .6rem 0;">
    To activate the <b>Tunneling Mechanic</b>, integrate the <code>tunneling()</code> function into the part of the game engine where the player’s hand is evaluated for a bust. 
    Instead of immediately declaring a loss when the total exceeds 21, the engine should call <code>tunneling()</code> to give the player a probabilistic second chance.
  </p>
  <ol style="margin:.2rem 0 0 1.4rem; padding:0;">
    <li>Locate the section of your function that checks:
      <pre style="background:#f1f5f9; padding:8px 10px; border-radius:6px; margin:.4rem 0;">if player_total &gt; 21:
    print("You busted! Dealer wins!")</pre>
    </li>
    <li>Replace that logic with:
      <pre style="background:#f1f5f9; padding:8px 10px; border-radius:6px; margin:.4rem 0;">if player_total &gt; 21:
    tunneling(player_total, p)</pre>
    </li>
    <li>Ensure that <code>tunneling()</code> returns the final hand value (if you modify it to do so), or updates the game state directly.</li>
    <li>Continue the normal game flow afterward — the dealer plays, and the round concludes according to the new total.</li>
  </ol>
  <p style="margin-top:.6rem;">
    This integration allows tunneling to serve as a small, rule-consistent “bonus layer” triggered only in bust scenarios, without affecting other parts of gameplay.
  </p>
</div>
